In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

def scrape_nobel_laureates():
    url = "https://en.wikipedia.org/wiki/List_of_Nobel_laureates"
    headers = {'User-Agent': 'Mozilla/5.0'}

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"Connection Error: {e}")
        return None

    soup = BeautifulSoup(response.text, 'lxml')

    # 1. Find the main wikitable
    table = soup.find('table', {'class': 'wikitable'})
    if not table:
        print("Error: Could not find the table.")
        return None

    # 2. Identify the Subject headers
    # We look at the first row to map which column is which subject
    header_row = table.find_all('tr')[0]
    # Extract text and filter out 'Year' to get the list of subjects
    cols = [th.get_text(strip=True) for th in header_row.find_all('th')]
    print(cols)

    # Standard subjects order usually: Physics, Chemistry, Medicine, Literature, Peace, Economics
    # We skip index 0 because it is 'Year'
    subjects = cols[1:]

    laureates_data = []

    # 3. Process each row
    rows = table.find_all('tr')[1:] # Skip header
    for row in rows:
        cells = row.find_all(['td', 'th'])
        if not cells:
            continue

        # Extract Year - Handling references like '1901[a]' using regex
        raw_year = cells[0].get_text(strip=True)
        year_match = re.search(r'\d{4}', raw_year) # Find the 4-digit year
        if not year_match:
            continue
        year = year_match.group()

        # 4. Iterate through the subject cells (starting from index 1)
        for i, cell in enumerate(cells[1:]):
            if i >= len(subjects):
                break

            current_subject = subjects[i]

            # Find all links (potential winners) in this cell
            links = cell.find_all('a', href=True)
            for link in links:
                name = link.get_text(strip=True)
                url = link['href']

                # Filter: Ignore empty names, citations [1], and the 'Year' column links
                if name and "/wiki/" in url and not url.startswith("#") and not name.isdigit():
                    laureates_data.append({
                        "winner_name": name,
                        "subject": current_subject,
                        "year": year,
                        "url": url
                    })

    # Create DataFrame
    df = pd.DataFrame(laureates_data)
    return df

# Execute and check
df = scrape_nobel_laureates()

if df is not None and not df.empty:
    # Display top 5 to match your exercise screenshot
    print(df.head(5).to_string())
else:
    print("DataFrame is still empty. Check the Wikipedia URL or table class.")

['Year', 'Physics', 'Chemistry', 'Physiologyor Medicine', 'Literature', 'Peace', 'Economic Sciences[13][a]']
                    winner_name                subject  year                                                         url
0               Wilhelm Röntgen                Physics  1901               https://en.wikipedia.org/wiki/Wilhelm_Röntgen
1  Jacobus Henricus van 't Hoff              Chemistry  1901  https://en.wikipedia.org/wiki/Jacobus_Henricus_van_'t_Hoff
2              Emil von Behring  Physiologyor Medicine  1901              https://en.wikipedia.org/wiki/Emil_von_Behring
3               Sully Prudhomme             Literature  1901               https://en.wikipedia.org/wiki/Sully_Prudhomme
4                  Henry Dunant                  Peace  1901                  https://en.wikipedia.org/wiki/Henry_Dunant
